# Урок 2.7. Закрепление: квиз и проект kb-search

**Модуль 2 «Embeddings и векторные базы данных» · Курс «LLM для Python-разработчиков»**

Финал модуля — по проверенному рецепту урока 1.8:

1. **Карта модуля** — что теперь в вашем арсенале и где это выстрелит.
2. **Сводный квиз** — 10 сценариев «как в жизни», без подглядывания.
3. **Проект** — CLI-утилита `kb-search`: настоящий поиск по базе знаний, собранный из всего, что вы прошли, — с двумя взаимозаменяемыми бэкендами.

⏱ **Время:** ~90–120 минут (большая часть — на самостоятельную попытку собрать утилиту).

## 1. Карта модуля: что у вас теперь есть

| урок | главный навык | где выстрелит дальше |
|---|---|---|
| 2.1 Эмбеддинги | текст → вектор, косинус, дистрибутивная гипотеза, e5-префиксы | вся retrieval-половина RAG (модуль 3) |
| 2.2 Векторные БД | ANN, recall@k, IVF и HNSW руками, ловушка фильтров | осознанный выбор и тюнинг любого векторного движка |
| 2.3 Chroma | коллекции, метрика, метаданные, distance vs similarity | прототипы и ноутбуки |
| 2.4 Qdrant | Docker, точки/payload, score, payload-индексы, снапшоты | продакшен-поиск, RAG-сервис модуля 3 |
| 2.5 Чанкинг | разрешение поиска, стратегии, паспорта чанков, двухступенчатость | качество каждого RAG |
| 2.6 Продвинутое | контекстные заголовки, semantic chunking, parent-document, cross-encoder | тюнинг ретривера по показаниям eval |

Сквозная линия модуля — **дисциплина замера**: каждое улучшение проверялось eval'ом, и трижды замер опровергал интуицию (few-shot в 1.7, чанкинг-точность в 2.5, диверсификация в 2.5). Эта привычка — самое ценное, что вы забираете в модуль 3.

## 2. Сводный квиз

Сценарии — как на архитектурном ревью. Сначала свой ответ, потом спойлер.

**1. Коллега: «Зачем нам эмбеддинги? У нас Elasticsearch с полнотекстовым поиском, работает отлично».** Когда он прав, а когда нет?

<details><summary>Ответ</summary>

Прав для точных совпадений: артикулы, коды ошибок, имена — keyword-поиск здесь сильнее. Не прав там, где запрос и документ говорят об одном разными словами: «отгул» против «отпуска», «упал прод» против «SEV1» — наш eval показывал этот провал лексики весь модуль. Зрелое решение — гибрид: keyword + векторный поиск дополняют друг друга.

</details>

**2. Команда перешла на новую модель эмбеддингов, а коллекцию не переиндексировала — «векторы же уже есть».** Что будет?

<details><summary>Ответ</summary>

Поиск-лотерея: запросы эмбеддятся новой моделью, документы лежат в координатах старой — это разные «карты смыслов», их косинусы бессмысленны (урок 2.1). Правило жёсткое: модель — часть схемы коллекции; сменили модель → переиндексировали всё.

</details>

**3. Прототип: 30 000 документов, 3 запроса в секунду. Тимлид требует кластер Qdrant с репликацией.** Ваши аргументы?

<details><summary>Ответ</summary>

На таком масштабе полный перебор numpy — единицы миллисекунд с recall 100% (урок 2.2), а Chroma даёт персистентность без единого контейнера. Qdrant оправдан, когда поиск становится сервисом: несколько клиентов, миллионы векторов, фильтры на масштабе, бэкапы. Преждевременная инфраструктура — это налог без выгоды.

</details>

**4. После включения ANN-индекса пользователи жалуются: «документ точно есть, а поиск его не находит».** Диагноз и лечение?

<details><summary>Ответ</summary>

Это цена ANN — потеря recall (урок 2.2): жадный обход графа промахнулся мимо истинного соседа. Диагноз ставится замером recall@k против полного перебора на выборке запросов; лечение — поднять `ef_search` (или `nprobe`), заплатив латентностью. Ручка, а не константа.

</details>

**5. Мигрировали с Chroma на Qdrant, и фильтр «отбрасывать результаты хуже 0.25» стал выкидывать лучшие находки.** Что произошло?

<details><summary>Ответ</summary>

Конвенции инвертированы (уроки 2.3–2.4): Chroma возвращает distance (меньше = ближе), Qdrant — score (больше = ближе). Порог «< 0.25 — хорошо» превратился в «< 0.25 — худшее». Пороги и сортировки пересматриваются при любой смене движка; ловится eval'ом до деплоя.

</details>

**6. Фильтр «только документы своего отдела» реализовали так: ищем топ-10, потом выкидываем чужое. У маленьких отделов выдача пустая.** Почему и как правильно?

<details><summary>Ответ</summary>

Классический post-filter (урок 2.2): если отделу принадлежит 5% базы, в топ-10 его документов почти не бывает. Правильно — фильтровать *во время* поиска: `query_filter` в Qdrant (filtered HNSW) или pre-filter подмножества; на крайний случай — oversampling с большим k.

</details>

**7. В индекс заливают страницы вики по 15 000 символов целиком, модель — e5-small.** Сколько текста реально ищется?

<details><summary>Ответ</summary>

Около десятой части: лимит e5-small — 512 токенов (~1000–1500 символов русского), остальное тихо отбрасывается при эмбеддинге (урок 2.5). Плюс вектор даже этого начала — «среднее по больнице». Лечение: чанкинг по структуре + контекстные заголовки, потолок размера — лимит модели.

</details>

**8. Включили диверсификацию `max_per_doc=1` — и вопросы, у которых оба ответа в одном документе, стали получать мусор вторым результатом.** Это баг?

<details><summary>Ответ</summary>

Нет, это тёмная сторона приёма (урок 2.5): диверсификация жертвует вторым лучшим чанком ради разнообразия — иногда зря. Разумный дефолт `max_per_doc=2`, а само значение — предмет eval'а на своих вопросах, не догма.

</details>

**9. «Давайте прикрутим semantic chunking к нашей базе markdown-документов с заголовками и списками».** Стоит ли?

<details><summary>Ответ</summary>

Скорее нет: у markdown структура уже размечена автором — заголовки и абзацы дают границы бесплатно (урок 2.5). Semantic chunking — эмбеддинг каждого предложения — оправдан для «каши» без структуры: расшифровки звонков, OCR, чаты (урок 2.6). Дешёвое сначала.

</details>

**10. Юридическая база: цена ошибки в топ-3 высока, +0.5 секунды латентности приемлемо.** Что добавить в пайплайн и во что это обойдётся?

<details><summary>Ответ</summary>

Cross-encoder второй ступенью (урок 2.6): bi-encoder приносит топ-20–30 кандидатов, cross-encoder перечитывает пары «вопрос+фрагмент». Бюджет сходится: ~25 мс × 30 пар ≈ 0.75 с на CPU (меньше на GPU) — в рамках. Для русского — мультиязычный `mmarco-mMiniLMv2`; включение — только после замера прироста на eval.

</details>

## 3. Проект: `kb-search`

Техзадание:

> CLI-утилита поиска по базе знаний «Векторики». Стандартный `argparse`, конфигурация из окружения. Команды: `health`, `index`, `search` (с `--topic` и `--diverse`), `eval`. Главное архитектурное требование — **два взаимозаменяемых бэкенда за одним интерфейсом**:

| | `lexical` | `semantic` |
|---|---|---|
| скоринг | TF-IDF (уроки 2.5–2.6) | e5 + косинус (урок 2.1) |
| хранилище | JSON-файл | Qdrant (урок 2.4) |
| зависимости | stdlib + utils | модель + сервер |
| синонимы | нет | да |

Режим `auto` выбирает semantic, когда модель в кэше и Qdrant отвечает, — иначе честно работает на lexical. Это не учебная условность, а продакшен-паттерн: поиск обязан деградировать изящно, а не падать, когда зависимость недоступна.

**Сначала — сами.** Всё нужное вы писали в уроках модуля: чанкер с заголовками — 2.5/2.6, TF-IDF — 2.5, Qdrant-клиент — 2.4, eval — везде. Готовая реализация: [`project/kb_search.py`](project/kb_search.py) (README рядом; в эталоне, помимо техзадания, реализованы и все задания для прокачки из раздела 6 — `compare`, `backup`, `--parent`, `--chunking semantic`, `--rerank`).

## 4. Разбор ключевых решений

**Один интерфейс — два бэкенда.** Оба класса реализуют `index(chunks)` и `search(query, top_k, topic)` — остальной код не знает, кто под капотом:

```python
def pick_backend(name: str):
    if name == "lexical":  return LexicalBackend()
    if name == "semantic": return SemanticBackend()
    try:
        return SemanticBackend()      # auto: пробуем лучший...
    except Exception:
        print("[semantic недоступен — работаю на lexical]", file=sys.stderr)
        return LexicalBackend()       # ...и честно деградируем
```

**Ленивые импорты.** `qdrant_client` и `sentence_transformers` импортируются *внутри* `SemanticBackend` — lexical-путь работает даже там, где эти пакеты не установлены. Утилита никогда не падает с `ImportError` на команде, которой зависимость не нужна.

**Индексируем одно, показываем другое** (урок 2.6): у чанка два поля — `indexed_text` (с контекстным заголовком, для скоринга) и `text` (оригинал, для показа). Роли не смешиваются.

**`--topic` — это pre-filter** (уроки 2.2/2.4): lexical-бэкенд пропускает чужие чанки до скоринга, semantic передаёт `query_filter` в Qdrant. Никаких пустых выдач post-фильтра.

**Детерминированные id** (урок 2.4): `uuid5(NAMESPACE_URL, chunk_id)` — одинаковые чанки всегда попадают в одни и те же точки. Но перед `upsert` коллекция **пересоздаётся**: другая стратегия чанкинга (`--chunking semantic`) даёт другое число чанков, и без пересоздания точки старой нарезки остались бы в индексе сиротами.

## 5. Принимаем работу

Утилиту запускаем как пользователь — отдельным процессом:

In [1]:
import subprocess
import sys
from pathlib import Path

CLI = Path("project") / "kb_search.py"
assert CLI.exists(), "Запустите ноутбук из папки 02-embeddings-vector-db"

def run_cli(*cli_args: str) -> None:
    result = subprocess.run(
        [sys.executable, str(CLI), *cli_args],
        capture_output=True, text=True, encoding="utf-8", timeout=600,
    )
    if result.stdout:
        print(result.stdout.rstrip())
    if result.stderr:
        print("--- stderr:", result.stderr.rstrip())
    print(f"--- exit code: {result.returncode}")

print("Хелпер готов. Утилита:", CLI)

Хелпер готов. Утилита: project\kb_search.py


In [2]:
# Диагностика: что доступно на этой машине прямо сейчас
run_cli("health")

[ OK ] данные: 14 документов в data/
[ OK ] Qdrant 1.19.0 на http://localhost:6333
[ OK ] модель intfloat/multilingual-e5-small в кэше
[ OK ] lexical-индекс построен: 31 чанков
[ OK ] cross-encoder cross-encoder/mmarco-mMiniLMv2-L12-H384-v1 в кэше — rerank ce доступен
[ OK ] Ollama 0.32.15 на http://127.0.0.1:11434 — rerank llm доступен
--- exit code: 0


In [3]:
# Индексация: чанкинг + контекстные заголовки + паспорта (уроки 2.5-2.6)
run_cli("index", "--chunking", "semantic")

[semantic] проиндексировано: 14 документов -> 42 чанков (чанкинг: semantic)
--- exit code: 0


In [4]:
# Точный вопрос - находится конкретный абзац с ответом, а не документ
run_cli("search", "Сколько компания доплачивает в первые дни болезни?", "--top", "2")

# Parent-window (урок 2.6): нашли точным чанком, отдаём окно +-300 символов
run_cli("search", "Сколько компания доплачивает в первые дни болезни?",
        "--top", "1", "--parent")

1. [0.880] Больничные и отсутствие по болезни  (bolnichny_2, тема: hr)
   ну: размер зависит от стажа. Компания доплачивает до 100% оклада первые 10 дней болезни, если стаж в «Векторике» больше года. Работать во время больничного запрещено — это требование трудового законод...

2. [0.869] Больничные и отсутствие по болезни  (bolnichny_1, тема: hr)
   Открывать официальный больничный лист (ЭЛН) для отсутствия до 3 рабочих дней не обязательно: в «Векторике» действует правило трёх дней доверия — до 3 дней болезни в квартал без больничного листа с сох...
--- exit code: 0
1. [0.880] Больничные и отсутствие по болезни  (bolnichny_2, тема: hr)
   чанк 210 симв. -> окно +-300: 503 симв.
   ...действует правило трёх дней доверия — до 3 дней болезни в квартал без больничного листа с сохранением оплаты.  При болезни дольше 3 дней оформляйте электронный больничный в поликлинике. Номер ЭЛН передайте в бухгалтерию через HR-портал, раздел «Документы».  Больничный оплачивается по закону: размер зависи

In [5]:
# Pre-filter по теме (уроки 2.2/2.4): ищем только в it-документах
run_cli("search", "Куда писать про сломанный ноутбук?", "--topic", "it", "--top", "2")

# Диверсификация (урок 2.5): двухтемный вопрос - выдача покрывает обе темы
run_cli("search", "Как оформить больничный и отпуск?", "--diverse", "--top", "3")

1. [0.819] Информационная безопасность  (security_1, тема: it)
   Секреты приложений (API-ключи, токены, строки подключения к базам) хранятся в HashiCorp Vault и попадают в приложение через переменные окружения при деплое. Коммит секрета в git — инцидент безопасност...

2. [0.818] Рабочее оборудование  (oborudovanie_1, тема: it)
   Личное оборудование для работы с кодом компании использовать запрещено политикой безопасности. Ремонт корпоративной техники — только через IT-отдел; самостоятельно вскрывать ноутбук нельзя, это снимае...
--- exit code: 0


1. [0.883] Больничные и отсутствие по болезни  (bolnichny_1, тема: hr)
   Больничный оплачивается по закону: размер зависит от стажа. Компания доплачивает до 100% оклада первые 10 дней болезни, если стаж в «Векторике» больше года.  Работать во время больничного запрещено — ...

2. [0.846] Отпуск в компании «Векторика»  (otpusk_0, тема: hr)
   Отпуск в компании «Векторика»  Каждому сотруднику положено 28 календарных дней оплачиваемого отпуска в год. Отпуск можно дробить на части, но одна из частей должна быть не короче 14 дней подряд.  Заяв...

3. [0.836] Инциденты и дежурства  (incidents_1, тема: dev)
   После каждого инцидента SEV1 и SEV2 в течение трёх рабочих дней пишется постмортем (post-mortem) по шаблону: хронология, причина, влияние, действия. Постмортемы безвинные (blameless) — ищем причину в ...
--- exit code: 0


In [ ]:
# Вторая ступень (урок 2.6): судья пересортировывает топ-10 первой ступени.
# Если cross-encoder не скачан или Ollama выключена - утилита предупредит
# в stderr и честно покажет выдачу первой ступени.
run_cli("search", "Какая надбавка за неделю дежурства?", "--rerank", "llm", "--top", "2")
run_cli("search", "Какая надбавка за неделю дежурства?", "--rerank", "ce", "--top", "2")

Вторая ступень: LLM-судья qwen2.5:3b (да=1/нет=0, ничьи — порядком первой ступени)

1. [1.000] Инциденты и дежурства  (incidents_1, тема: dev)
   После каждого инцидента SEV1 и SEV2 в течение трёх рабочих дней пишется постмортем (post-mortem) по шаблону: хронология, причина, влияние, действия. Постмортемы безвинные (blameless) — ищем причину в ...

2. [0.000] Инциденты и дежурства  (incidents_0, тема: dev)
   Инциденты и дежурства  Дежурство (on-call) длится одну неделю и передаётся по понедельникам в 12:00. Дежурный инженер получает алерты в Telegram и PagerDuty и обязан отреагировать в течение 15 минут в...
--- exit code: 0


Вторая ступень: cross-encoder (счёт — логит, бывает отрицательным)

1. [2.838] Инциденты и дежурства  (incidents_1, тема: dev)
   После каждого инцидента SEV1 и SEV2 в течение трёх рабочих дней пишется постмортем (post-mortem) по шаблону: хронология, причина, влияние, действия. Постмортемы безвинные (blameless) — ищем причину в ...

2. [-2.390] Удалённая работа и гибридный график  (udalenka_0, тема: hr)
   Удалённая работа и гибридный график  «Векторика» работает в гибридном формате: минимум 2 дня в неделю в офисе (вторник и четверг — общие офисные дни), остальные дни — по желанию.  Полностью удалённый ...
--- exit code: 0


In [7]:
# Встроенный eval - те же 7 вопросов, что сопровождали нас весь модуль
run_cli("eval")

# И таблица-сравнение бэкендов: один eval, две колонки, никаких впечатлений
run_cli("compare")


[+] Сколько дней отпуска положено в году?
[+] Как оформить больничный лист?
[+] VPN подключён, но внутренние сайты не открываются
[+] Кто должен посмотреть мой код перед мержем?
[~] Упал прод, что делать в первую очередь?
[+] Сколько компания доплачивает в первые дни болезни?
[+] За сколько минут дежурный должен отреагировать ночью?

[semantic] top-1: 6/7   top-3: 7/7   средняя находка: 393 симв.
--- exit code: 0


вопрос                                                    lexical  semantic
Сколько дней отпуска положено в году?                           +         +
Как оформить больничный лист?                                   +         +
VPN подключён, но внутренние сайты не открываются               +         +
Кто должен посмотреть мой код перед мержем?                     +         +
Упал прод, что делать в первую очередь?                         x         ~
Сколько компания доплачивает в первые дни болезни?              +         +
За сколько минут дежурный должен отреагировать ночью?           +         +
---------------------------------------------------------------------------
top-1                                                         6/7       6/7
top-3                                                         6/7       7/7
средняя находка, симв.                                        375       393
время eval, с                                                0.02      0.84
--- exit cod

Разбор приёмки (у вас цифры могут чуть отличаться — зависят от того, что доступно на машине):

- На этой машине e5 в кэше и Qdrant запущен, поэтому `auto` выбрал **semantic**: 6/7 top-1 и **7/7 top-3** со средней находкой ~393 символа. Единственный не-top-1 — «Упал прод»: даже эмбеддинги не сразу связывают его с «SEV1 — сервис недоступен», но в top-3 документ есть. Лексика здесь промахивается полностью — это видно в таблице `compare`.
- **`compare` — один eval, две колонки.** Качество на мини-базе почти одинаковое (обе 6/7 top-1), вся разница — в строке «Упал прод» (x против ~) и в цене: lexical отрабатывает за сотые доли секунды, semantic — за ~1.5 с, потому что каждый запрос гоняет нейросеть. Такие таблицы и заканчивают споры «какой бэкенд лучше».
- **`--parent`** показал механику parent-document из урока 2.6: попали точным чанком в 242 символа, а отдали окно в 538 — LLM следующего модуля получит и ответ, и контекст вокруг него.
- **`--rerank`** — вторая ступень пересортировала топ-10: LLM-судья ответил «да» только чанку с «надбавка 15%» (1.000 против 0.000), cross-encoder дал ему логит +2.8 при −2…−4 у остальных — зазор между релевантным и мусором и есть его почерк. Судья здесь бинарный и оценивает кандидатов по одному: батч-шкала 0–10 из решения 3 урока 2.6 у 3B-модели развалилась в живом прогоне (подробности — в задании 5 ниже).
- Если моделей на машине нет, тот же CLI без единой правки честно живёт на lexical (предупреждение о деградации — в stderr, результаты — в stdout; утилиту можно встраивать в скрипты — привычка из урока 1.8), а `health` подскажет, чего именно не хватает.


## 6. Задания для прокачки

По возрастанию сложности:

1. **`compare`** — команда, гоняющая eval на обоих бэкендах и печатающая таблицу-сравнение (после скачивания e5 станет самой показательной командой утилиты).
2. **`--parent`** — флаг для `search`: отдавать родительский документ или окно ±300 символов вокруг находки (урок 2.6, решение 2 — почти готовый код).
3. **`--chunking semantic`** — альтернативная стратегия индексации для владельцев e5 (урок 2.6, акт 2): границы по долинам смысловой кривой.
4. **`backup`** — снапшот коллекции Qdrant через API (урок 2.4; помните `wait=False` и терпение на Windows).
5. **`rerank`** — вторая ступень: cross-encoder (урок 2.6) или LLM-судья (промпт `build_rerank_prompt` уже написан в решении 3 урока 2.6 — останется подключить Ollama из модуля 1).

Эталонная реализация **всех пяти** уже лежит в `project/kb_search.py`: команды `compare` и `backup`, флаги `--parent [window|full]`, `--chunking semantic` и `--rerank ce|llm`. Порядок работы прежний — **сначала сами, не подглядывая**: напишите свою версию, прогоните `eval`, и только потом сравните с эталоном. Обе rerank-ступени честно деградируют, если модели нет: `ce` требует скачать cross-encoder (~500 МБ), `llm` — запущенную Ollama из модуля 1.

И грабли из эталона, найденные живым прогоном: батч-промпт `build_rerank_prompt` со шкалой 0–10 (решение 3 урока 2.6) у 3B-судьи разваливается — в нашем прогоне он поставил 10 больничному на вопрос про упавший прод. Надёжен только **бинарный да/нет по одному кандидату** (это же выяснится в уроке 3.3), причём вопрос должен стоять **после** фрагмента, ближе к месту решения (урок 1.2): с вопросом в начале тот же судья отвечал «нет» даже дословным совпадениям. Итоговый промпт — в `build_judge_prompt` эталона.


## 🎓 Итоги модуля 2

Вы прошли путь от «что такое вектор» до работающего поискового сервиса:

- **эмбеддинги** — «хеш, сохраняющий смысл»; косинус; дистрибутивная гипотеза, проверенная руками;
- **индексы** — IVF и малый мир HNSW, написанные с нуля, с честной ценой recall;
- **две БД** — Chroma для прототипов, Qdrant для прода, и все инвертированные конвенции между ними;
- **чанкинг** — разрешение поиска, паспорта фрагментов, заголовки, родители;
- **двухступенчатый поиск** — recall дёшево, precision дорого;
- **eval** — рефлекс, который трижды спас нас от красивых, но вредных идей.

**Что дальше — модуль 3 «RAG: Retrieval-Augmented Generation».** Посмотрите на `kb-search search "..."` — он возвращает *фрагменты*. А пользователь хочет *ответ*: «Компания доплачивает до 100% оклада первые 10 дней, если стаж больше года». Осталось соединить две половины курса: ваш поиск (модуль 2) находит контекст → ваша LLM (модуль 1) пишет по нему ответ. Это и есть RAG — и у вас уже готовы обе половины.

## 📚 Что почитать

- [Qdrant: Vector Search Manuals](https://qdrant.tech/articles/) — статьи команды Qdrant: от квантизации до гибридного поиска
- [Pinecone Learning Center](https://www.pinecone.io/learn/) — лучшая бесплатная библиотека статей по векторному поиску
- [MTEB Leaderboard](https://huggingface.co/spaces/mteb/leaderboard) — когда захотите модель сильнее e5-small
- [BEIR benchmark](https://github.com/beir-cellar/beir) — как индустрия меряет retrieval (вы уже делаете то же самое, только меньше)
- README проекта: [`project/README.md`](project/README.md) — команды, конфигурация, идеи